<a href="https://colab.research.google.com/github/krishadesai16/autonomous-financial-intelligence-agent/blob/main/Autonomous_AI_Financial_%26_Market_Intelligence_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q yfinance transformers torch gradio plotly pandas

In [2]:
# Ingestion and FinBERT NLP Setup
import yfinance as yf
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Pipeline running on: {device}")

# Load FinBERT
MODEL_NAME = "ProsusAI/finbert"
print("[*] Downloading & loading FinBERT transformer weights...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

LABEL_MAP = {0: "Positive", 1: "Negative", 2: "Neutral"}
print("[✓] NLP Engine initialized successfully.")

[*] Pipeline running on: cuda
[*] Downloading & loading FinBERT transformer weights...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

[✓] NLP Engine initialized successfully.


In [ ]:
#  Searchable Dropdown + Real-Time FinBERT Agent Dashboard
import urllib.request
import xml.etree.ElementTree as ET
import plotly.graph_objects as go
import gradio as gr

# Pre-populated catalog of popular Global and Indian equities
COMPANY_CATALOG = [
    ("NVIDIA Corporation (NVDA)", "NVDA"),
    ("Apple Inc. (AAPL)", "AAPL"),
    ("Microsoft Corporation (MSFT)", "MSFT"),
    ("Tesla Inc. (TSLA)", "TSLA"),
    ("Alphabet / Google (GOOGL)", "GOOGL"),
    ("Reliance Industries (RELIANCE.NS)", "RELIANCE.NS"),
    ("Tata Consultancy Services (TCS.NS)", "TCS.NS"),
    ("HDFC Bank (HDFCBANK.NS)", "HDFCBANK.NS"),
    ("Infosys Limited (INFY.NS)", "INFY.NS"),
    ("State Bank of India (SBIN.NS)", "SBIN.NS"),
    ("ICICI Bank (ICICIBANK.NS)", "ICICIBANK.NS"),
    ("Tata Motors (TATAMOTORS.NS)", "TATAMOTORS.NS")
]

def fetch_realtime_headlines(ticker_symbol: str, max_items: int = 8):
    headlines = []

    # 1. Primary: Google News Financial RSS Feed
    try:
        encoded_query = f"{ticker_symbol}+stock+market"
        url = f"https://news.google.com/rss/search?q={encoded_query}&hl=en-US&gl=US&ceid=US:en"

        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=5) as response:
            xml_data = response.read()

        root = ET.fromstring(xml_data)
        for item in root.findall('./channel/item')[:max_items]:
            title = item.find('title').text
            if title:
                clean_title = title.split(" - ")[0].strip()
                headlines.append(clean_title)
    except Exception as e:
        print(f"[!] RSS fetch notice: {e}")

    # 2. Fallback to yfinance if RSS returns empty
    if not headlines:
        try:
            ticker = yf.Ticker(ticker_symbol)
            raw_news = ticker.news or []
            for item in raw_news[:max_items]:
                t = item.get("title")
                if not t and "content" in item and isinstance(item["content"], dict):
                    t = item["content"].get("title")
                if t:
                    headlines.append(t.strip())
        except Exception:
            pass

    # 3. Default safeguard
    if not headlines:
        headlines = [
            f"Market movements and earnings forecasts for {ticker_symbol} remain aligned with standard analyst guidance.",
            f"Institutional trading desks report steady volume for {ticker_symbol} during regular sessions."
        ]

    return headlines

def analyze_market_asset(selected_input: str):
    if not selected_input:
        return "### ⚠️ Error: Please select or enter a company ticker.", None, None, None

    # Extraction logic: If user typed/selected from catalog "Company Name (TICKER)", extract TICKER
    selected_input = selected_input.strip()
    if "(" in selected_input and ")" in selected_input:
        ticker_symbol = selected_input.split("(")[-1].split(")")[0].strip().upper()
    else:
        ticker_symbol = selected_input.upper()

    # 1. Quantitative Price Ingestion
    try:
        ticker = yf.Ticker(ticker_symbol)
        hist = ticker.history(period="1mo")
        if hist.empty:
            return f"### ⚠️ Error: No price data found for '{ticker_symbol}'. Verify ticker symbol.", None, None, None
    except Exception as e:
        return f"### ⚠️ Extraction Error: {str(e)}", None, None, None

    current_price = float(hist['Close'].iloc[-1])
    sma_20 = float(hist['Close'].rolling(window=20).mean().iloc[-1])
    month_return = float(((current_price - hist['Close'].iloc[0]) / hist['Close'].iloc[0]) * 100)

    # 2. Real-Time Qualitative News Ingestion
    extracted_headlines = fetch_realtime_headlines(ticker_symbol, max_items=6)

    # 3. FinBERT NLP Inference Pass
    inputs = tokenizer(
        extracted_headlines,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = F.softmax(outputs.logits, dim=-1).cpu().numpy()

    table_data = []
    polarities = []

    for i, title in enumerate(extracted_headlines):
        pos_prob = float(probabilities[i][0])
        neg_prob = float(probabilities[i][1])
        neu_prob = float(probabilities[i][2])
        label = LABEL_MAP[int(probabilities[i].argmax())]

        pol = pos_prob - neg_prob
        polarities.append(pol)

        table_data.append({
            "Headline": title,
            "FinBERT Sentiment": label,
            "Polarity Score": f"{pol:+.2f}"
        })

    avg_polarity = sum(polarities) / len(polarities)
    sentiment_df = pd.DataFrame(table_data)

    # 4. Multi-Factor Decision Fusion
    if avg_polarity > 0.10 and current_price >= sma_20:
        stance = "BULLISH (BUY / OUTPERFORM)"
        badge_color = "#16a34a"
        rationale = "Positive narrative news flow aligns with upward momentum above the 20-day moving average."
    elif avg_polarity < -0.10 and current_price < sma_20:
        stance = "BEARISH (REDUCE / HEDGE)"
        badge_color = "#dc2626"
        rationale = "Negative press coverage combined with downward momentum below the 20-day SMA signals elevated downside risk."
    else:
        stance = "NEUTRAL / HOLD"
        badge_color = "#ca8a04"
        rationale = "Divergent technical and sentiment indicators suggest market consolidation; avoid aggressive position sizing."

    # 5. Executive Markdown Brief
    brief_md = f"""
    ### 📊 Executive Intelligence Brief: **{ticker_symbol}**

    * **Current Price:** `{current_price:.2f}` (30-Day Return: `{month_return:+.2f}%`)
    * **Technical Momentum (20-Day SMA):** `{sma_20:.2f}` ({'Trading Above SMA' if current_price >= sma_20 else 'Trading Below SMA'})
    * **Aggregate Sentiment Polarity:** `{avg_polarity:+.2f}` (Scale: -1.0 Bearish to +1.0 Bullish)

    <div style="padding: 12px; border-radius: 8px; background-color: {badge_color}15; border-left: 5px solid {badge_color}; margin: 10px 0;">
        <span style="font-size: 1.1em; font-weight: bold; color: {badge_color};">Agent Stance: {stance}</span><br>
        <span style="color: #374151;"><strong>Thesis:</strong> {rationale}</span>
    </div>
    """

    # 6. Interactive Visual: Price & SMA
    fig_price = go.Figure()
    fig_price.add_trace(go.Scatter(
        x=hist.index, y=hist['Close'],
        mode='lines+markers', name='Closing Price',
        line=dict(color='#2563eb', width=2)
    ))
    fig_price.add_trace(go.Scatter(
        x=hist.index, y=hist['Close'].rolling(20).mean(),
        mode='lines', name='20-Day SMA',
        line=dict(color='#f59e0b', width=2, dash='dash')
    ))
    fig_price.update_layout(
        title=f"{ticker_symbol} — 30-Day Trajectory vs. 20-Day SMA",
        xaxis_title="Date",
        yaxis_title="Asset Price",
        template="plotly_white",
        margin=dict(l=20, r=20, t=40, b=20),
        height=320
    )

    # 7. Interactive Visual: Sentiment Radial Gauge
    fig_gauge = go.Figure(go.Indicator(
        mode="gauge+number",
        value=avg_polarity,
        domain={'x': [0, 1], 'y': [0, 1]},
        title={'text': "Market Sentiment Index"},
        gauge={
            'axis': {'range': [-1, 1], 'tickwidth': 1, 'tickcolor': "black"},
            'bar': {'color': "#1e293b"},
            'steps': [
                {'range': [-1, -0.10], 'color': '#fee2e2'},
                {'range': [-0.10, 0.10], 'color': '#fef9c3'},
                {'range': [0.10, 1], 'color': '#dcfce7'}
            ],
            'threshold': {
                'line': {'color': badge_color, 'width': 4},
                'thickness': 0.75,
                'value': avg_polarity
            }
        }
    ))
    fig_gauge.update_layout(height=320, margin=dict(l=20, r=20, t=40, b=20))

    return brief_md, fig_price, fig_gauge, sentiment_df

# Build Dashboard UI
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", neutral_hue="slate")) as app:
    gr.Markdown("# 🤖 Autonomous AI Financial & Market Intelligence Agent")
    gr.Markdown("Real-time equity intelligence powered by **FinBERT NLP** transformer sentiment and quantitative momentum.")

    with gr.Row():
        company_dropdown = gr.Dropdown(
            choices=[label for label, _ in COMPANY_CATALOG],
            value="NVIDIA Corporation (NVDA)",
            label="Select Company / Search or Type Any Ticker",
            allow_custom_value=True,
            filterable=True,
            scale=3
        )
        submit_btn = gr.Button("Analyze Market Signals", variant="primary", scale=1)

    brief_output = gr.Markdown()

    with gr.Row():
        chart_price_output = gr.Plot(label="Price Trajectory & Trend")
        chart_gauge_output = gr.Plot(label="Sentiment Meter")

    gr.Markdown("### 📰 FinBERT Granular News Dissection")
    table_output = gr.Dataframe(headers=["Headline", "FinBERT Sentiment", "Polarity Score"], wrap=True)

    submit_btn.click(
        fn=analyze_market_asset,
        inputs=company_dropdown,
        outputs=[brief_output, chart_price_output, chart_gauge_output, table_output]
    )

app.launch(share=True, debug=True)

/tmp/ipykernel_3318/418961735.py:203: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.



Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ad27c8a32a80dc0530.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
